# Pipeline Financeiro — inform_27 + Danone
Carrega dados de transporte, associa ingressos Danone, enriquece com coordenadas e quilometragem, calcula métricas de rentabilidade e exporta para Parquet (Power BI).

## 1. Configuração e ligação à base de dados

In [1]:
import platform
import sqlite3
import warnings

import pandas as pd

warnings.filterwarnings("ignore")


def get_paths() -> dict:
    """Devolve os caminhos de ficheiros consoante o sistema operativo."""
    sistema = platform.system()

    if sistema == "Windows":
        return {
            "db": r"C:\Users\LISARR\Documents\python\00.DB\2026.db",
            "parquet": r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27_2026_final.parquet",
            "caminho_excel": r"C:\Users\LISARR\Documents\python\01.Financeiro\Danone_Custos_V2.xlsx",
        }

    if sistema == "Darwin":
        icloud = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen"
        return {
            "db": f"{icloud}/00_DB/2026.db",
            "parquet": f"{icloud}/inform_27_2026_final.parquet",
            "caminho_excel": f"{icloud}/Danone_Custos_v2.xlsx",
        }

    return {
        "db": "2026.db",
        "parquet": "inform_27_2026_final.parquet",
        "excel": "Danone_Custos_v2.xlsx",
    }


PATHS = get_paths()

with sqlite3.connect(PATHS["db"]) as con:
    df = pd.read_sql_query("SELECT * FROM inform_27_2026", con)

print(f"Linhas carregadas: {len(df):,}")
df.head()

df_excel = pd.read_excel(PATHS["caminho_excel"])
print(f"Linhas Excel: {len(df_excel):,}")
df_excel.head()

Linhas carregadas: 774,239
Linhas Excel: 38,973


,BLIINF,CODACT,PREFPE,PFEENT,PCODCL,PNOMCL,PRUTA,PCATCL,CODTLI,CODMOP,...,Unnamed: 27,Unnamed: 28,Tarifa,Ingreso,Combustible,Month,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36
0,FGE50PTG,11,5021758091 413963878,20260102,350150908.0,PREÇO BAIXO - DOIS AMIGOS,1734,PRE,STD,MAS,...,Porto Prevenda,5021758091 413963878,31.96,1.307803,1.399123,1,NaN,NaN,Month,NaN
1,FGE50PTG,11,5021765321 413964675,20260102,350151650.0,PREÇO BAIXO SUP. AV. FER. AROS,1784,PRE,STD,MAS,...,Porto Prevenda,5021765321 413964675,31.96,1.186355,1.269195,1,NaN,NaN,01,131504.440575
2,FGE50PTG,11,5021758092 413969049,20260102,350392489.0,"SANDRA TROVISCO, UNIPESSOAL, L",1734,PRE,STD,MAS,...,Porto Prevenda,5021758092 413969049,31.96,0.714626,0.764526,1,NaN,NaN,02,120918.450190
3,FGE50PTG,11,5021742353 413966257,20260102,350194778.0,PREÇO BAIXO - ANGEIRAS,1734,PRE,STD,MAS,...,Porto Prevenda,5021742353 413966257,31.96,0.884653,0.946426,1,NaN,NaN,03,142470.218474
4,FGE50PTG,11,5021756292 413963116,20260102,350390901.0,FROIZ - BRAGA II- RETAIL CENTE,1734,PRE,STD,MAS,...,Porto Prevenda,5021756292 413963116,31.96,3.466382,3.708429,1,NaN,NaN,04,144627.303661


## 2. Dados base — `inform_27_2026`

In [2]:
# Conversão de tipos numéricos
colunas_numericas = [
    "INGRESODT", "COSTEDT", "RENTADT", "PALETSDT",
    "PESO_BRUTO", "PALETS", "KM", "KMREALES",
]
for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

df["CODEUT"] = df["CODEUT"].astype(str)

# Remover espaços em branco de todas as colunas de texto
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

# Filtrar GESTION == "LIS", ou GESTION nulo com PROPIETARIO em CEP/PTG
df = df[
    (df["GESTION"] == "LIS")
    | (df["GESTION"].isna() & df["PROPIETARIO"].isin(["CEP", "PTG"]))
]

# Datas (após o trim, sem componente de hora)
df["FCARGA"] = pd.to_datetime(df["FCARGA"], format="%Y%m%d", errors="coerce").dt.date
df["FENTREGA"] = pd.to_datetime(df["FENTREGA"], format="%Y%m%d", errors="coerce").dt.date

df.head()

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,LOCDES,LUGARDESCARGA,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem
93,94,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,158.09,-158.09,11.00,...,300390,NO USAR TRANP.FERNANDO SIMÕES MONTEIRO,RFG,EUR,Trailer 33 plts,33,DIESEL,145.559,NO,SAL_DAT027 (1).xls
94,95,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,5.46,-5.46,0.38,...,319558,3. MALAQUIAS - CASH & CARRY O. AZ,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,230.939,NO,SAL_DAT027 (1).xls
95,96,PTG,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,13.3,6.75,6.55,0.47,...,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,TAM,EUR,Trailer 33 plts,33,DIESEL,151.803,NO,SAL_DAT027 (1).xls
96,97,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,15.95,-15.95,1.11,...,319783,"3. MARABUTO-PRODUT.ALIMENTARES,SA",RFGRFG,EUR,Trailer 33 plts,33,DIESEL,204.542,NO,SAL_DAT027 (1).xls
97,98,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,7.33,-7.33,0.51,...,319201,3. COOPERATIVA AGRICOLA DA TOCHA,RFGRFG,EUR,Trailer 33 plts,33,DIESEL,171.983,NO,SAL_DAT027 (1).xls


## 3. Pipeline Danone — cálculo do ingresso por entrega

In [3]:
# FORK: ingresso_danone vem do ficheiro Excel (KILOSPO), não da BD
caminho_excel = PATHS["caminho_excel"]

raw_kilospo = pd.read_excel(caminho_excel, sheet_name="KILOSPO", header=None)
df_danone = raw_kilospo.iloc[1:].copy()
df_danone.columns = raw_kilospo.iloc[0]

df_danone["PREFPE"] = df_danone["PREFPE"].astype("string").str.strip()

# PFEENT pode vir como texto "20260102" ou já como data (Timestamp), consoante o ambiente.
# Normalizar sempre para o formato AAAAMMDD em texto.
if pd.api.types.is_datetime64_any_dtype(df_danone["PFEENT"]):
    df_danone["PFEENT"] = df_danone["PFEENT"].dt.strftime("%Y%m%d")
else:
    df_danone["PFEENT"] = df_danone["PFEENT"].astype("string").str.strip()

df_danone["PCODCL"] = df_danone["PCODCL"].astype("string").str.strip()
df_danone["tipo_local"] = df_danone["Ruta WH"].astype("string").str.strip()
df_danone["ingresso_danone"] = pd.to_numeric(df_danone["Combustible"], errors="coerce")

print(f"Linhas carregadas da KILOSPO: {len(df_danone):,}")
df_danone[["PREFPE", "PFEENT", "PCODCL", "PNOMCL", "tipo_local", "ingresso_danone"]].head()

Linhas carregadas da KILOSPO: 38,973


,PREFPE,PFEENT,PCODCL,PNOMCL,tipo_local,ingresso_danone
1,5021758091 413963878,20260102,350150908,PREÇO BAIXO - DOIS AMIGOS,Porto Prevenda,1.399123
2,5021765321 413964675,20260102,350151650,PREÇO BAIXO SUP. AV. FER. AROS,Porto Prevenda,1.269195
3,5021758092 413969049,20260102,350392489,"SANDRA TROVISCO, UNIPESSOAL, L",Porto Prevenda,0.764526
4,5021742353 413966257,20260102,350194778,PREÇO BAIXO - ANGEIRAS,Porto Prevenda,0.946426
5,5021756292 413963116,20260102,350390901,FROIZ - BRAGA II- RETAIL CENTE,Porto Prevenda,3.708429


In [4]:
df_danone.head(5)

,BLIINF,CODACT,PREFPE,PFEENT,PCODCL,PNOMCL,PRUTA,PCATCL,CODTLI,CODMOP,...,Tarifa,Ingreso,Combustible,Month,NaN,NaN,NaN,NaN,tipo_local,ingresso_danone
1,FGE50PTG,011,5021758091 413963878,20260102,350150908,PREÇO BAIXO - DOIS AMIGOS,1734,PRE,STD,MAS,...,31.96,1.307803,1.399123,01,NaN,NaN,Month,NaN,Porto Prevenda,1.399123
2,FGE50PTG,011,5021765321 413964675,20260102,350151650,PREÇO BAIXO SUP. AV. FER. AROS,1784,PRE,STD,MAS,...,31.96,1.186355,1.269195,01,NaN,NaN,01,131504.440575,Porto Prevenda,1.269195
3,FGE50PTG,011,5021758092 413969049,20260102,350392489,"SANDRA TROVISCO, UNIPESSOAL, L",1734,PRE,STD,MAS,...,31.96,0.714626,0.764526,01,NaN,NaN,02,120918.450190,Porto Prevenda,0.764526
4,FGE50PTG,011,5021742353 413966257,20260102,350194778,PREÇO BAIXO - ANGEIRAS,1734,PRE,STD,MAS,...,31.96,0.884653,0.946426,01,NaN,NaN,03,142470.218474,Porto Prevenda,0.946426
5,FGE50PTG,011,5021756292 413963116,20260102,350390901,FROIZ - BRAGA II- RETAIL CENTE,1734,PRE,STD,MAS,...,31.96,3.466382,3.708429,01,NaN,NaN,04,144627.303661,Porto Prevenda,3.708429


In [5]:
sem_classificacao = df_danone[df_danone["tipo_local"].isna()]

lista_referencias_sem_classificacao = (
    sem_classificacao[["PCODCL", "PNOMCL", "PREFPE", "PFEENT"]]
    .sort_values(["PCODCL", "PFEENT"])
    .reset_index(drop=True)
)

print(f"Linhas sem tipo_local: {len(lista_referencias_sem_classificacao):,}")
lista_referencias_sem_classificacao

Linhas sem tipo_local: 269


,PCODCL,PNOMCL,PREFPE,PFEENT
0,350244321,"JNR - DISTRIBUICAO ALIMENTAR,",5021958305 414184555,20260209
1,350334015,SPAR - ARMAÇAO PERA,5023278112 415414768,20260708
2,350334015,SPAR - ARMAÇAO PERA,5023341922 415474572,20260715
3,350334015,SPAR - ARMAÇAO PERA,5023372448 415501427,20260722
4,350334015,SPAR - ARMAÇAO PERA,5023453164 415577274,20260729
...,...,...,...,...
264,350494128,RENATO ZUMACH RIBEIRO-PO842282,5023387039 415566895,20260728
265,350494129,RENATO ZUMACH RIBEIRO-PO842282,5023454209 415581067,20260730
266,350494415,FORÇA DE VENCER RIO MOINHOS,5023476080 415604284,20260731
267,350494415,FORÇA DE VENCER RIO MOINHOS,5023457132 415613004,20260801


In [6]:
df_danone["PFEENT"] = pd.to_datetime(df_danone["PFEENT"], format="%Y%m%d", errors="coerce")
df_danone["mes"] = df_danone["PFEENT"].dt.month

soma_por_mes = (
    df_danone
    .groupby("mes")
    .agg(ingresso_total=("ingresso_danone", lambda x: x.sum(min_count=1)))
    .reset_index()
)
soma_por_mes["ingresso_total"] = soma_por_mes["ingresso_total"].round(0).map("{:,.0f}".format)
soma_por_mes

,mes,ingresso_total
0,1,"131,504"
1,2,"120,918"
2,3,"142,470"
3,4,"144,627"
4,5,"142,736"
5,6,"155,498"
6,7,"180,669"
7,8,"10,704"


In [7]:
import calendar

mes8 = df_danone[df_danone["mes"] == 8]

dias_com_dados = mes8["PFEENT"].dt.day.nunique()
dias_total_mes = calendar.monthrange(2026, 8)[1]
total_mes8 = mes8["ingresso_danone"].sum(min_count=1)
media_diaria = total_mes8 / dias_com_dados
projecao_mes8 = media_diaria * dias_total_mes

print(f"Dias com dados em agosto: {dias_com_dados}")
print(f"Dias totais no mês: {dias_total_mes}")
print(f"Total registado até agora: {total_mes8:,.0f}")
print(f"Média diária: {media_diaria:,.0f}")
print(f"Projeção para o mês completo: {projecao_mes8:,.0f}")

Dias com dados em agosto: 2
Dias totais no mês: 31
Total registado até agora: 10,704
Média diária: 5,352
Projeção para o mês completo: 165,917


## 4. Associar o ingresso Danone ao dataframe principal

In [8]:
# Somar ingresso Danone por entrega (PREFPE + PFEENT)
df_ingresso_danone = df_danone[["PREFPE", "PFEENT", "ingresso_danone"]].copy()

# PREFPE fica inteiro (não se divide por espaços) — REFERENCIA no df guarda
# os dois códigos colados no mesmo formato ("5021758091     413963878"),
# por isso a chave de correspondência tem de ficar igual dos dois lados.
df_ingresso_danone["PREFPE"] = df_ingresso_danone["PREFPE"].astype("string").str.strip()

# PFEENT pode já vir como data (Timestamp) ou como texto "AAAAMMDD", consoante a fonte.
if pd.api.types.is_datetime64_any_dtype(df_ingresso_danone["PFEENT"]):
    df_ingresso_danone["PFEENT"] = pd.to_datetime(df_ingresso_danone["PFEENT"])
else:
    df_ingresso_danone["PFEENT"] = pd.to_datetime(
        df_ingresso_danone["PFEENT"].astype("string").str.strip(), format="%Y%m%d", errors="coerce"
    )

df_ingresso_danone = (
    df_ingresso_danone
    .groupby(["PREFPE", "PFEENT"], dropna=False)
    .agg(ingresso_danone_total=("ingresso_danone", lambda x: x.sum(min_count=1)))
    .reset_index()
)

# Preparar chaves no df principal
df["REFERENCIA"] = df["REFERENCIA"].astype("string").str.strip()
df["FENTREGA"] = pd.to_datetime(df["FENTREGA"], errors="coerce")
df["_ordem_original"] = range(len(df))

df = df.merge(
    df_ingresso_danone,
    left_on=["REFERENCIA", "FENTREGA"],
    right_on=["PREFPE", "PFEENT"],
    how="left",
    validate="many_to_one",
).sort_values("_ordem_original").reset_index(drop=True)

# Uma entrega pode ter várias linhas em df (mesma REFERENCIA + FENTREGA);
# o ingresso só é atribuído à última linha, para não o contar em duplicado.
ultima_linha = ~df.duplicated(subset=["REFERENCIA", "FENTREGA"], keep="last")

df["ingresso_danone"] = pd.Series(pd.NA, index=df.index, dtype="Float64")
df.loc[ultima_linha, "ingresso_danone"] = df.loc[ultima_linha, "ingresso_danone_total"]

df = df.drop(columns=["PREFPE", "PFEENT", "ingresso_danone_total", "_ordem_original"])

# Ingresso total = ingresso do transporte (INGRESODT) + ingresso Danone
df["total_ingresso"] = df["INGRESODT"].fillna(0) + df["ingresso_danone"].fillna(0)

diferenca = df_danone["ingresso_danone"].sum(min_count=1) - df["ingresso_danone"].sum(min_count=1)
print(f"Diferença entre ingresso Danone calculado e associado: {diferenca:,.2f}")

Diferença entre ingresso Danone calculado e associado: 9,169.97


In [9]:
df["mes"] = df["FENTREGA"].dt.month

soma_por_mes_df = (
    df
    .groupby("mes")
    .agg(ingresso_total=("ingresso_danone", lambda x: x.sum(min_count=1)))
    .reset_index()
)
soma_por_mes_df["ingresso_total"] = soma_por_mes_df["ingresso_total"].round(0).map("{:,.0f}".format)
soma_por_mes_df

,mes,ingresso_total
0,1,"130,574"
1,2,"120,179"
2,3,"141,883"
3,4,"143,911"
4,5,"142,139"
5,6,"152,179"
6,7,"178,415"
7,8,"10,680"


In [10]:
resumo_por_atividade_mes = (
    df[df["CODACT"].isin(["011", "013", "311"])]
    .groupby(["CODACT", "mes"])
    .agg(
        soma_ingresodt=("INGRESODT", "sum"),
        soma_ingresso_danone=("ingresso_danone", lambda x: x.sum(min_count=1)),
    )
    .reset_index()
)

resumo_por_atividade_mes["soma_ingresodt"] = resumo_por_atividade_mes["soma_ingresodt"].round(0).map("{:,.0f}".format)
resumo_por_atividade_mes["soma_ingresso_danone"] = resumo_por_atividade_mes["soma_ingresso_danone"].round(0).map("{:,.0f}".format)

resumo_por_atividade_mes

,CODACT,mes,soma_ingresodt,soma_ingresso_danone
0,011,1,"82,711","126,268"
1,011,2,"71,846","115,959"
2,011,3,"82,752","136,841"
3,011,4,"79,561","138,418"
4,011,5,"84,288","136,997"
5,011,6,"80,993","146,036"
6,011,7,"36,231","170,888"
7,011,8,"72,116","10,423"
8,013,1,0,"4,306"
9,013,2,0,"4,220"


## 5. Features derivadas

In [11]:
# Data, semana e dia da semana
df["data"] = pd.to_datetime(df["FCARGA"])
df["week_number"] = df["data"].dt.isocalendar().week
df["week_day"] = df["data"].dt.day_name()
df["mes"] = df["data"].dt.month
df["mes_nome"] = df["data"].dt.month_name()

df[["FCARGA", "week_number", "week_day", "mes_nome", "total_ingresso"]].head()

,FCARGA,week_number,week_day,mes_nome,total_ingresso
0,2026-02-01,5,Sunday,February,0.0
1,2026-02-01,5,Sunday,February,33.520821
2,2026-02-01,5,Sunday,February,13.3
3,2026-02-01,5,Sunday,February,112.762292
4,2026-02-01,5,Sunday,February,45.421036


In [12]:
# Normalização da capacidade do camião (agrupar capacidades equivalentes)
mapeamento_capacidade = {
    4: 6, 5: 6, 6: 6,
    8: 12, 12: 12,
    14: 20, 15: 20, 16: 20, 18: 20, 20: 20,
    22: 24, 24: 24,
    33: 33, 66: 66,
}

df["CAMION_CAPACIDAD_NUM"] = pd.to_numeric(df["CAMION_CAPACIDAD"], errors="coerce")
df["capacidade_norm"] = df["CAMION_CAPACIDAD_NUM"].map(mapeamento_capacidade)

# Capacidade em falta (valor 0): preencher com a capacidade mais comum da mesma rota
linhas_sem_capacidade = df["CAMION_CAPACIDAD_NUM"] == 0
if linhas_sem_capacidade.any():
    capacidade_por_rota = (
        df.loc[df["CAMION_CAPACIDAD_NUM"] > 0]
        .groupby("CODEUT")["capacidade_norm"]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.max())
    )
    for rota in df.loc[linhas_sem_capacidade, "CODEUT"].unique():
        if rota in capacidade_por_rota.index:
            df.loc[(df["CODEUT"] == rota) & linhas_sem_capacidade, "capacidade_norm"] = capacidade_por_rota[rota]

df["dados_validos"] = df["capacidade_norm"].notna()
print(f"Linhas com capacidade válida: {df['dados_validos'].sum():,} / {len(df):,}")

Linhas com capacidade válida: 118,078 / 122,923


## 6. Coordenadas geográficas (origem e destino)

In [13]:
with sqlite3.connect(PATHS["db"]) as con:
    coords = pd.read_sql_query(
        "SELECT cp AS CP, point_x AS POINT_X, point_y AS POINT_Y FROM coordenadas",
        con,
    )

coords["CP"] = coords["CP"].astype("string").str.strip()
coords["POINT_X"] = pd.to_numeric(coords["POINT_X"], errors="coerce")
coords["POINT_Y"] = pd.to_numeric(coords["POINT_Y"], errors="coerce")
coords["CP_parte"] = coords["CP"].str.split("-").str[0]

# Centroide por prefixo de código postal — usado como fallback quando o CP completo não tem match
centroid = (
    coords.groupby("CP_parte")[["POINT_X", "POINT_Y"]]
    .mean()
    .reset_index()
    .rename(columns={"POINT_X": "longitude_centroid", "POINT_Y": "latitude_centroid"})
)

coords_origem = coords[["CP", "POINT_X", "POINT_Y"]].rename(
    columns={"CP": "CPOSTAL", "POINT_X": "longitude_origem", "POINT_Y": "latitude_origem"}
)
coords_destino = coords[["CP", "POINT_X", "POINT_Y"]].rename(
    columns={"CP": "CPOSTAD", "POINT_X": "longitude_destino", "POINT_Y": "latitude_destino"}
)

df["CPOSTAL_parte"] = df["CPOSTAL"].str.split("-").str[0]
df["CPOSTAD_parte"] = df["CPOSTAD"].str.split("-").str[0]

df = df.merge(coords_origem, on="CPOSTAL", how="left")
df = df.merge(coords_destino, on="CPOSTAD", how="left")

# Preencher falhas de match com o centroide do prefixo do código postal
centroid_origem = centroid.rename(columns={
    "CP_parte": "CPOSTAL_parte",
    "longitude_centroid": "longitude_origem_c",
    "latitude_centroid": "latitude_origem_c",
})
df = df.merge(centroid_origem, on="CPOSTAL_parte", how="left")
df["longitude_origem"] = df["longitude_origem"].fillna(df["longitude_origem_c"])
df["latitude_origem"] = df["latitude_origem"].fillna(df["latitude_origem_c"])

centroid_destino = centroid.rename(columns={
    "CP_parte": "CPOSTAD_parte",
    "longitude_centroid": "longitude_destino_c",
    "latitude_centroid": "latitude_destino_c",
})
df = df.merge(centroid_destino, on="CPOSTAD_parte", how="left")
df["longitude_destino"] = df["longitude_destino"].fillna(df["longitude_destino_c"])
df["latitude_destino"] = df["latitude_destino"].fillna(df["latitude_destino_c"])

df = df.drop(columns=[
    "CPOSTAL_parte", "CPOSTAD_parte",
    "longitude_origem_c", "latitude_origem_c",
    "longitude_destino_c", "latitude_destino_c",
])

print(f"Matches origem: {df['longitude_origem'].notna().sum():,}")
print(f"Matches destino: {df['longitude_destino'].notna().sum():,}")


Matches origem: 0
Matches destino: 0


## 7. Métricas de rentabilidade

In [14]:
df["custo_por_palete"] = df["COSTEDT"] / df["PALETS"].replace(0, 1)
df["ingresso_por_palete"] = df["total_ingresso"] / df["PALETS"].replace(0, 1)

# Taxa de ocupação por rota (CODEUT): total de paletes da rota vs capacidade do veículo
rota_totais = (
    df.groupby("CODEUT")
    .agg(total_palets=("PALETS", "sum"), capacidade_rota=("capacidade_norm", "first"))
    .reset_index()
)
rota_totais["taxa_rota"] = rota_totais["total_palets"] / rota_totais["capacidade_rota"].replace(0, 1) * 100

df = df.merge(rota_totais[["CODEUT", "taxa_rota", "total_palets"]], on="CODEUT", how="left")
df["taxa_ocupacao"] = df["PALETS"] / df["total_palets"].replace(0, 1) * df["taxa_rota"]
df = df.drop(columns=["taxa_rota", "total_palets"])

df["margem"] = df["total_ingresso"] - df["COSTEDT"]
df["margem_por_palete"] = df["ingresso_por_palete"] - df["custo_por_palete"]

df.head()

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,dados_validos,longitude_origem,latitude_origem,longitude_destino,latitude_destino,custo_por_palete,ingresso_por_palete,taxa_ocupacao,margem,margem_por_palete
0,94,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,158.09,-158.09,11.00,...,True,NaN,NaN,NaN,NaN,14.371818,0.0,33.333333,-158.09,-14.371818
1,95,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,5.46,-5.46,0.38,...,True,NaN,NaN,NaN,NaN,5.460000,33.520821,3.030303,28.060821,28.060821
2,96,PTG,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,13.3,6.75,6.55,0.47,...,True,NaN,NaN,NaN,NaN,6.750000,13.3,3.030303,6.55,6.55
3,97,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,15.95,-15.95,1.11,...,True,NaN,NaN,NaN,NaN,7.975000,56.381146,6.060606,96.812292,48.406146
4,98,CEP,SALVESEN LOGISTICA PORTUGAL / SALVESEN LOGISTI...,Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,L153182,0.0,7.33,-7.33,0.51,...,True,NaN,NaN,NaN,NaN,7.330000,45.421036,3.030303,38.091036,38.091036


## 8. Selecionar colunas finais e exportar

In [15]:
colunas_finais = [
    "REFERENCIA", "FENTREGA", "CODEUT", "capacidade_norm", "PALETS", "INGRESODT", "ingresso_danone","total_ingresso", "COSTEDT",
    "PROV_ORIGEN", "LOCORIGEN", "PROV_DESTINO", "LOCDESTINO", "CPOSTAL", "CPOSTAD",
    "longitude_origem", "latitude_origem", "longitude_destino", "latitude_destino",
    "TRANSPORTISTA", "week_day", "week_number", "mes", "mes_nome", "data",
    "dados_validos", "LOCCAR", "LUGARCARGA", "LOCDES", "LUGARDESCARGA", "TRACTORA",
    "PESO_BRUTO", "CODEDT", "CODACT", "TIPOCLIENTE", "TIPOFLUJO",
    "PROV_ENTREGAR", "PAISENTREGAR", "ACTIVIDAD",
    "custo_por_palete", "ingresso_por_palete", "taxa_ocupacao", "margem", "margem_por_palete",
]

df_sel = df[colunas_finais].copy()
df_sel.head()

,REFERENCIA,FENTREGA,CODEUT,capacidade_norm,PALETS,INGRESODT,ingresso_danone,total_ingresso,COSTEDT,PROV_ORIGEN,...,TIPOCLIENTE,TIPOFLUJO,PROV_ENTREGAR,PAISENTREGAR,ACTIVIDAD,custo_por_palete,ingresso_por_palete,taxa_ocupacao,margem,margem_por_palete
0,COIMBRA 20/29+70/79-02.02,2026-02-02,3365589,33.0,11.0,0.0,<NA>,0.0,158.09,Lisboa,...,None,Directo,ANADIA,PORTUGAL,DANONE PORTUGAL CAPILAR,14.371818,0.0,33.333333,-158.09,-14.371818
1,5021950798 414190635,2026-02-02,3365589,33.0,1.0,0.0,33.520821,33.520821,5.46,Lisboa,...,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL,5.460000,33.520821,3.030303,28.060821,28.060821
2,2021792046,2026-02-02,3365589,33.0,1.0,13.3,<NA>,13.3,6.75,Lisboa,...,None,Directo,Coimbra,PORTUGAL,SUMOLCOMPAL MARKETING,6.750000,13.3,3.030303,6.55,6.55
3,5021982094 414203894,2026-02-02,3365589,33.0,2.0,0.0,112.762292,112.762292,15.95,Lisboa,...,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL,7.975000,56.381146,6.060606,96.812292,48.406146
4,5021963392 414182550,2026-02-02,3365589,33.0,1.0,0.0,45.421036,45.421036,7.33,Lisboa,...,TLD PORTUGAL,Directo,Coimbra,PORTUGAL,DANONE PORTUGAL,7.330000,45.421036,3.030303,38.091036,38.091036


In [21]:
df_sel.to_parquet(PATHS["parquet"], index=False)

print(f"Ficheiro exportado: {PATHS['parquet']}")
print(f"Linhas: {len(df_sel):,}")
print(f"Colunas: {len(df_sel.columns)}")
print(f"Tamanho: {df_sel.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Ficheiro exportado: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27_2026_final.parquet
Linhas: 122,923
Colunas: 44
Tamanho: 179.50 MB


# Analises


In [23]:
# Marcar entregas com ingresso_danone atribuído (fazem match ao Excel Danone)
df["tem_danone"] = df.groupby(["REFERENCIA", "FENTREGA"])["ingresso_danone"].transform(lambda x: x.notna().any())

resumo_danone_mes = (
    df[df["CODACT"].isin(["011"]) & df["tem_danone"]]
    .groupby("mes")
    .agg(
        ingresso=("ingresso_danone", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_danone_mes["margem"] = resumo_danone_mes["ingresso"] - resumo_danone_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_danone_mes[coluna] = resumo_danone_mes[coluna].round(0).map("{:,.0f}".format)

resumo_danone_mes

,mes,ingresso,custo,paletes,peso,margem
0,1,"125,529","117,786","14,603","4,648,770","7,744"
1,2,"115,325","109,520","12,705","3,956,792","5,805"
2,3,"141,272","135,447","14,836","5,177,255","5,825"
3,4,"140,306","132,134","14,928","4,666,314","8,172"
4,5,"133,963","132,184","15,610","4,940,692","1,779"
5,6,"150,979","140,485","15,115","5,253,169","10,493"
6,7,"163,724","151,078","15,520","5,481,699","12,646"
7,8,"4,330","4,081",439,"134,973",248
8,12,"6,402","5,933",499,"153,533",469


In [27]:
resumo_311_mes = (
    df[df["CODACT"].isin(["311"])]
    .groupby("mes")
    .agg(
        ingresso=("INGRESODT", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_311_mes["margem"] = resumo_311_mes["ingresso"] - resumo_311_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_311_mes[coluna] = resumo_311_mes[coluna].round(0).map("{:,.0f}".format)

resumo_311_mes

,mes,ingresso,custo,paletes,peso,margem
0,1,"19,652","28,412","3,435","1,079,834","-8,760"
1,2,"18,675","24,506","3,210","832,053","-5,831"
2,3,"20,237","24,005","3,165","955,649","-3,768"
3,4,"23,730","28,626","3,758","1,136,324","-4,896"
4,5,"19,382","23,490","3,208","879,063","-4,108"
5,6,"19,142","32,696","3,371","1,004,582","-13,554"
6,7,"22,879","31,623","3,930","1,262,652","-8,744"
7,8,"16,682","22,686","3,293","874,893","-6,003"
8,12,"1,869","2,735",244,"83,981",-866


In [28]:
resumo_outros_mes = (
    df[~df["CODACT"].isin(["011", "311", "013"])]
    .groupby("mes")
    .agg(
        ingresso=("INGRESODT", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_outros_mes["margem"] = resumo_outros_mes["ingresso"] - resumo_outros_mes["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_outros_mes[coluna] = resumo_outros_mes[coluna].round(0).map("{:,.0f}".format)

resumo_outros_mes

,mes,ingresso,custo,paletes,peso,margem
0,1,"290,785","251,345","27,317","4,578,603","39,441"
1,2,"262,043","222,249","22,278","4,099,397","39,793"
2,3,"317,548","291,166","29,595","6,946,743","26,382"
3,4,"321,175","302,946","28,947","5,694,049","18,230"
4,5,"320,619","304,797","27,493","5,762,710","15,822"
5,6,"316,347","299,691","26,457","5,848,740","16,656"
6,7,"384,483","331,431","28,232","6,425,722","53,052"
7,8,"282,338","260,154","22,649","5,397,828","22,185"
8,12,"20,853","20,484","1,476","333,839",369


In [29]:
resumo_total_sem_013 = (
    df[~df["CODACT"].isin(["013"])]
    .groupby("mes")
    .agg(
        ingresso=("total_ingresso", "sum"),
        custo=("COSTEDT", "sum"),
        paletes=("PALETS", "sum"),
        peso=("PESO_BRUTO", "sum"),
    )
    .reset_index()
)
resumo_total_sem_013["margem"] = resumo_total_sem_013["ingresso"] - resumo_total_sem_013["custo"]

for coluna in ["ingresso", "custo", "margem", "paletes", "peso"]:
    resumo_total_sem_013[coluna] = resumo_total_sem_013[coluna].round(0).map("{:,.0f}".format)

resumo_total_sem_013

,mes,ingresso,custo,paletes,peso,margem
0,1,"517,785","401,222","45,632","10,360,268","116,563"
1,2,"466,867","357,953","38,389","8,928,579","108,914"
2,3,"565,471","452,768","47,830","13,118,146","112,703"
3,4,"566,098","467,728","48,020","11,555,668","98,371"
4,5,"556,729","467,791","46,899","11,624,558","88,938"
5,6,"561,145","486,666","46,101","12,169,237","74,479"
6,7,"611,562","536,096","49,044","13,321,385","75,466"
7,8,"371,222","416,150","41,451","10,375,008","-44,928"
8,12,"33,889","29,280","2,222","572,885","4,608"
